0_Data Extraction

In [1]:
import pandas as pd

df = pd.read_csv("UCI_Credit_Card.csv")

print(df.shape)
df.head()

(30000, 25)


,ID,LIMIT_BAL,SEX,EDUCATION,MARRIAGE,AGE,PAY_1,PAY_2,PAY_3,PAY_4,...,BILL_AMT4,BILL_AMT5,BILL_AMT6,PAY_AMT1,PAY_AMT2,PAY_AMT3,PAY_AMT4,PAY_AMT5,PAY_AMT6,DEFAULT
0,1,20000.0,2,2,1,24,2,2,-1,-1,...,0.0,0.0,0.0,0.0,689.0,0.0,0.0,0.0,0.0,1
1,2,120000.0,2,2,2,26,-1,2,0,0,...,3272.0,3455.0,3261.0,0.0,1000.0,1000.0,1000.0,0.0,2000.0,1
2,3,90000.0,2,2,2,34,0,0,0,0,...,14331.0,14948.0,15549.0,1518.0,1500.0,1000.0,1000.0,1000.0,5000.0,0
3,4,50000.0,2,2,1,37,0,0,0,0,...,28314.0,28959.0,29547.0,2000.0,2019.0,1200.0,1100.0,1069.0,1000.0,0
4,5,50000.0,1,2,1,57,-1,0,-1,0,...,20940.0,19146.0,19131.0,2000.0,36681.0,10000.0,9000.0,689.0,679.0,0


In [3]:
print("Shape:", df.shape)

print("\nColumns:")
print(df.columns.tolist())

print("\nData types:")
print(df.dtypes)

print("\nMissing values:")
print(df.isnull().sum())

print("\nTarget distribution:")
print(df["DEFAULT"].value_counts())

Shape: (30000, 25)

Columns:
['ID', 'LIMIT_BAL', 'SEX', 'EDUCATION', 'MARRIAGE', 'AGE', 'PAY_1', 'PAY_2', 'PAY_3', 'PAY_4', 'PAY_5', 'PAY_6', 'BILL_AMT1', 'BILL_AMT2', 'BILL_AMT3', 'BILL_AMT4', 'BILL_AMT5', 'BILL_AMT6', 'PAY_AMT1', 'PAY_AMT2', 'PAY_AMT3', 'PAY_AMT4', 'PAY_AMT5', 'PAY_AMT6', 'DEFAULT']

Data types:
ID             int64
LIMIT_BAL    float64
SEX            int64
EDUCATION      int64
MARRIAGE       int64
AGE            int64
PAY_1          int64
PAY_2          int64
PAY_3          int64
PAY_4          int64
PAY_5          int64
PAY_6          int64
BILL_AMT1    float64
BILL_AMT2    float64
BILL_AMT3    float64
BILL_AMT4    float64
BILL_AMT5    float64
BILL_AMT6    float64
PAY_AMT1     float64
PAY_AMT2     float64
PAY_AMT3     float64
PAY_AMT4     float64
PAY_AMT5     float64
PAY_AMT6     float64
DEFAULT        int64
dtype: object

Missing values:
ID           0
LIMIT_BAL    0
SEX          0
EDUCATION    0
MARRIAGE     0
AGE          0
PAY_1        0
PAY_2        0
PAY_3   

The dataset has 30,000 observations and 25 variables, and no missing values are found. 

1 EDA&DATA PREPROCESSING

1.1 Load X&Y

In [10]:
X = df.drop(columns=["ID", "DEFAULT"])
y = df["DEFAULT"]

1.2 One-hot encode categorical variables

In [11]:
from sklearn.preprocessing import OneHotEncoder

cat_cols = ["SEX", "EDUCATION", "MARRIAGE"]

encoder = OneHotEncoder(sparse_output=False, handle_unknown="ignore")
encoded = encoder.fit_transform(X[cat_cols])

encoded_df = pd.DataFrame(
    encoded,
    columns=encoder.get_feature_names_out(cat_cols),
    index=X.index
)

X = pd.concat([X.drop(columns=cat_cols), encoded_df], axis=1)


1.3 Split into train and test sets

In [12]:
from sklearn.model_selection import train_test_split

X_train_full, X_test, y_train_full, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

1.4 Split training data into train and validation sets

In [14]:
X_train, X_val, y_train, y_val = train_test_split(
    X_train_full, y_train_full, test_size=1/8, random_state=42, stratify=y_train_full
)

1.5 Standardize the features

In [16]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()

X_train = scaler.fit_transform(X_train)
X_val = scaler.transform(X_val)
X_test = scaler.transform(X_test)

In [17]:
print(X_train.shape, X_val.shape, X_test.shape)


(21000, 33) (3000, 33) (6000, 33)


In the preprocessing stage, ID was removed and DEFAULT was used as the target variable, while categorical variables were transformed by one-hot encoding. The dataset was then divided into training, validation, and test sets, and all features were standardized, resulting in input matrices of size (21000, 33), (3000, 33), and (6000, 33) respectively.

2 Model Training

2.1 Logistic Regression

In [20]:
from sklearn.linear_model import LogisticRegression
lr_model = LogisticRegression(max_iter=1000, random_state=42)
lr_model.fit(X_train, y_train)

y_pred_lr = lr_model.predict(X_test)

y_prob_lr = lr_model.predict_proba(X_test)[:, 1]


In [22]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score
lr_accuracy = accuracy_score(y_test, y_pred_lr)
lr_recall = recall_score(y_test, y_pred_lr)
lr_precision = precision_score(y_test, y_pred_lr)
lr_auc = roc_auc_score(y_test, y_prob_lr)

print("Accuracy:", lr_accuracy)
print("Recall:", lr_recall)
print("Precision:", lr_precision)
print("AUC:", lr_auc)


Accuracy: 0.8095
Recall: 0.25018839487565936
Precision: 0.6916666666666667
AUC: 0.7105646750375862


The LR model achieved an accuracy of 0.8095, a recall of 0.25018839487565936, a precision of 0.6916666666666667, and an AUC of 0.7105646750375862 on the test set. 

2.2 SVM

In [24]:
from sklearn.svm import SVC
svm_model = SVC(kernel="rbf", random_state=42)
svm_model.fit(X_train, y_train)

y_pred_svm = svm_model.predict(X_test)

In [26]:
from sklearn.metrics import accuracy_score, recall_score, precision_score

svm_accuracy = accuracy_score(y_test, y_pred_svm)
svm_recall = recall_score(y_test, y_pred_svm)
svm_precision = precision_score(y_test, y_pred_svm)


print("Accuracy:", svm_accuracy)
print("Recall:", svm_recall)
print("Precision:", svm_precision)

Accuracy: 0.8158333333333333
Recall: 0.33685003767897514
Precision: 0.6651785714285714


The SVM model achieved an accuracy of 0.8158, a recall of 0.3369, and a precision of 0.6652 on the test set. 

2.3 Feedforward Neural Network

In [27]:
import torch
import torch.nn as nn
import torch.optim as optim

X_train_tensor = torch.tensor(X_train, dtype=torch.float32)
X_val_tensor = torch.tensor(X_val, dtype=torch.float32)
X_test_tensor = torch.tensor(X_test, dtype=torch.float32)

y_train_tensor = torch.tensor(y_train.values, dtype=torch.float32).view(-1, 1)
y_val_tensor = torch.tensor(y_val.values, dtype=torch.float32).view(-1, 1)
y_test_tensor = torch.tensor(y_test.values, dtype=torch.float32).view(-1, 1)

In [28]:
class FNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 64),
            nn.ReLU(),
            nn.Linear(64, 32),
            nn.ReLU(),
            nn.Linear(32, 1)
        )

    def forward(self, x):
        return self.net(x)

model = FNN(X_train.shape[1])

criterion = nn.BCEWithLogitsLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

num_epochs = 100
best_val_loss = float("inf")
best_model_state = None

for epoch in range(num_epochs):
    model.train()
    outputs = model(X_train_tensor)
    loss = criterion(outputs, y_train_tensor)

    optimizer.zero_grad()
    loss.backward()
    optimizer.step()

    model.eval()
    with torch.no_grad():
        val_outputs = model(X_val_tensor)
        val_loss = criterion(val_outputs, y_val_tensor)

    if val_loss.item() < best_val_loss:
        best_val_loss = val_loss.item()
        best_model_state = model.state_dict()

model.load_state_dict(best_model_state)

model.eval()
with torch.no_grad():
    y_logits = model(X_test_tensor)
    y_prob_nn = torch.sigmoid(y_logits).numpy().flatten()
y_pred_nn = (y_prob_nn >= 0.5).astype(int)


In [29]:
from sklearn.metrics import accuracy_score, recall_score, precision_score, roc_auc_score
nn_accuracy = accuracy_score(y_test, y_pred_nn)
nn_recall = recall_score(y_test, y_pred_nn)
nn_precision = precision_score(y_test, y_pred_nn)
nn_auc = roc_auc_score(y_test, y_prob_nn)

print("Accuracy:", nn_accuracy)
print("Recall:", nn_recall)
print("Precision:", nn_precision)
print("AUC:", nn_auc)

Accuracy: 0.8038333333333333
Recall: 0.2788244159758855
Precision: 0.6271186440677966
AUC: 0.7194223384960436


The feedforward neural network achieved an accuracy of 0.8038, a recall of 0.2788, a precision of 0.6271, and an AUC of 0.7194 on the test set. 

3 Compare Models

In [30]:
import pandas as pd

results = pd.DataFrame({
    "Model": ["Logistic Regression", "SVM", "Feedforward Neural Network"],
    "Accuracy": [lr_accuracy, svm_accuracy, nn_accuracy],
    "Recall": [lr_recall, svm_recall, nn_recall],
    "Precision": [lr_precision, svm_precision, nn_precision],
    "AUC": [lr_auc, None, nn_auc]
})

results


,Model,Accuracy,Recall,Precision,AUC
0,Logistic Regression,0.809500,0.250188,0.691667,0.710565
1,SVM,0.815833,0.336850,0.665179,NaN
2,Feedforward Neural Network,0.803833,0.278824,0.627119,0.719422


SVM performed best in terms of accuracy and recall, while logistic regression achieved the highest precision. The feedforward neural network produced the highest AUC, indicating a slightly better overall discrimination ability. 

Therefore, SVM is preferable when identifying more default cases is the main objective, whereas the neural network shows stronger overall ranking performance, and logistic regression remains the most interpretable model.

4 Which variables are most important for predicting defaults?

4.1 Get variable importance from logistic regression coefficients

In [31]:
feature_names = X.columns
importance_df = pd.DataFrame({
    "Feature": feature_names,
    "Coefficient": lr_model.coef_[0]
})

importance_df["Absolute_Coefficient"] = importance_df["Coefficient"].abs()
importance_df = importance_df.sort_values("Absolute_Coefficient", ascending=False)

importance_df.head(10)


,Feature,Coefficient,Absolute_Coefficient
2,PAY_1,0.668909,0.668909
8,BILL_AMT1,-0.315387,0.315387
15,PAY_AMT2,-0.201441,0.201441
14,PAY_AMT1,-0.188716,0.188716
10,BILL_AMT3,0.138146,0.138146
27,EDUCATION_5,-0.134259,0.134259
22,EDUCATION_0,-0.119560,0.119560
4,PAY_3,0.116704,0.116704
0,LIMIT_BAL,-0.116303,0.116303
3,PAY_2,0.097303,0.097303


The most important variables were identified based on the absolute values of the logistic regression coefficients. 

Among all predictors, PAY_1 was the most influential variable, indicating that the most recent repayment status played the strongest role in predicting default.

Other important variables included BILL_AMT1, PAY_AMT2, PAY_AMT1, BILL_AMT3, PAY_3, LIMIT_BAL, and PAY_2, suggesting that recent repayment behavior, bill amounts, payment amounts, and credit limit were all important factors in default prediction.